* link to dataset: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95/data_preview

In [ ]:
import pandas as pd


from helpers import summarize_columns


df = pd.read_csv(
    "datasets/Motor_Vehicle_Collisions_-_Crashes_20260407.csv", delimiter=","
)

summarize_columns(df)

## Discarding invalid rows & Redundant Columns

In [ ]:
# --- Removing Lat/Lon NaN values ---#
_rows_before = df.shape[0]
df = df[df["LONGITUDE"].notna() & df["LATITUDE"].notna()]
print(f"-> Dropped {_rows_before - df.shape[0]} rows after removing lat/lon NaNs rows")


# --- Removing redundant columns ---#
df = df.drop(["ZIP CODE", "LOCATION"], axis=1)


# --- Removing incidents with no injuries or deaths --- #
df = df[
    df["NUMBER OF PERSONS INJURED"].notna() | df["NUMBER OF PERSONS KILLED"].notna()
]

# summarize_columns(df)

## Fixing missing BUROUGHS

In [ ]:
import geopandas as gpd
from geodatasets import get_path

# -- Title Entries ex: BRONX -> Bronx
df["BOROUGH"] = df["BOROUGH"].str.title()


print(
    "Unique Boroughs include",
    list(df["BOROUGH"].unique()),
    "\n\n\t - Number of rows with missing 'BOROUGH' but present coordinates is:",
    sum(df["BOROUGH"].isna()),
)

_missing_mask = df["BOROUGH"].isna()

_df_missing = df.loc[_missing_mask, ["LONGITUDE", "LATITUDE"]]

_gdf_missing = gpd.GeoDataFrame(
    _df_missing,
    geometry=gpd.points_from_xy(
        _df_missing["LONGITUDE"], _df_missing["LATITUDE"]
    ),  # Function to create the mandatory 'geometry' column for a GeoDataFrame
    crs="EPSG:4326",  # Coordinate Reference System
)

# --- Using geodatasets to source boroughs multipoligon ---#
_path_to_boroughs = get_path("nybb")  # Literally path to the library chache
_boroughs_gdf = gpd.read_file(_path_to_boroughs)

_boroughs_gdf = _boroughs_gdf.to_crs(
    "EPSG:4326"
)  # Convert to standard coordinate reference system


# --- Left spacial join gdf_missing with boroughs multi-poligon ---#
_joined = gpd.sjoin(
    _gdf_missing,
    _boroughs_gdf[["BoroName", "geometry"]],
    how="left",
    predicate="within",
)

df.loc[_missing_mask, "BOROUGH"] = _joined["BoroName"]

print(
    "\n\t - Remaining missing BOROUGH after spatial fill:",
    df["BOROUGH"].isna().sum(),
    "Filled :",
    _gdf_missing.shape[0] - df["BOROUGH"].isna().sum(),
)

# --- Removing rows with BOROUGH == nan, after the fill ---#
df = df[df["BOROUGH"].notna()]

# summarize_columns(df)
del _missing_mask, _df_missing, _gdf_missing, _path_to_boroughs, _boroughs_gdf, _joined

## Memory Optimization

In [ ]:
df["CRASH DATETIME"] = pd.to_datetime(
    df["CRASH DATE"] + " " + df["CRASH TIME"], format="%m/%d/%Y %H:%M"
)

df = df.drop(["CRASH TIME", "CRASH DATE"], axis=1)


# --- Reducing memory footprint of numberic columns ---#
numeric_columns = [  # Small numeric values
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
    "NUMBER OF PEDESTRIANS INJURED",
    "NUMBER OF PEDESTRIANS KILLED",
    "NUMBER OF CYCLIST INJURED",
    "NUMBER OF CYCLIST KILLED",
    "NUMBER OF MOTORIST INJURED",
    "NUMBER OF MOTORIST KILLED",
]

for c in numeric_columns:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int8")


# --- Reducing memory footprint of categorical columns ---#
category_columns = [
    "CONTRIBUTING FACTOR VEHICLE 1",
    "CONTRIBUTING FACTOR VEHICLE 2",
    "CONTRIBUTING FACTOR VEHICLE 3",
    "CONTRIBUTING FACTOR VEHICLE 4",
    "CONTRIBUTING FACTOR VEHICLE 5",
    "VEHICLE TYPE CODE 1",
    "VEHICLE TYPE CODE 2",
    "VEHICLE TYPE CODE 3",
    "VEHICLE TYPE CODE 4",
    "VEHICLE TYPE CODE 5",
]

df = df.astype({k: "category" for k in category_columns})


# --- Reorder columns --- #
df = df[
    ["COLLISION_ID", "BOROUGH", "LATITUDE", "LONGITUDE", "CRASH DATETIME"]
    + numeric_columns
    + category_columns
    + ["ON STREET NAME", "CROSS STREET NAME", "OFF STREET NAME"]
]

summarize_columns(df)
print(df.columns)

In [ ]:
df = df.reset_index(drop=True)
df.to_parquet("datasets/crash_data.parquet", engine="pyarrow")

In [ ]:
pd.Series(df["COLLISION_ID"].unique()).to_csv(
    "datasets/collision_ids.csv", index=False, header=["COLLISION_ID"]
)
del df